# 🔍 AIOps Day 2 — Log Parsing với Drain3

**Họ và tên:** Lê Kim Dung  
**Ngày nộp:** 02/06/2026  
**Repository:** https://github.com/KimDung1/aiops_lekimdung

---

## Mục lục
1. [Setup & Synthetic Log Generation](#1-setup)
2. [Drain3 Log Parsing & Template Discovery](#2-drain3)
3. [Tuning: sim_th sweep](#3-tuning)
4. [Template Count Time Series & Anomaly Detection](#4-timeseries)
5. [New Template Detection](#5-new-template)
6. [Reflection: Metric vs Log](#6-reflection)
7. [Knowledge Check — Ảnh Viết Tay](#7-knowledge-check)


---
## 1. Setup & Synthetic Log Generation
<a id='1-setup'></a>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re, random, os, json
from collections import Counter
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)
os.makedirs('images', exist_ok=True)

print('✅ Setup complete')


In [ ]:
# ── Drain3 Simulator (lightweight implementation) ──
WILDCARD = '<*>'

class DrainSimulator:
    """
    Simplified Drain3 implementation.
    Drain uses a prefix tree (length → first_token → templates)
    and merges log lines that share similarity >= sim_th.
    """
    def __init__(self, sim_th=0.5):
        self.sim_th = sim_th
        self.root = {}           # (length, first_token) -> list of (template_tokens, count)
        self.template_counts = Counter()
        self.parse_records = []
        self.new_template_events = []

    def _tokenize(self, line):
        return re.sub(r'\d+', '<NUM>', line).split()

    def _similarity(self, tokens, template):
        if len(tokens) != len(template): return 0.0
        match = sum(1 for t, p in zip(tokens, template) if t == p or p == WILDCARD)
        return match / len(template)

    def _merge(self, tokens, template):
        return [t if t == p else WILDCARD for t, p in zip(tokens, template)]

    def parse(self, line, timestamp=None):
        tokens = self._tokenize(line)
        if not tokens: return None
        key = (len(tokens), tokens[0])
        bucket = self.root.setdefault(key, [])

        best_sim, best_idx = 0, -1
        for i, (tmpl, _) in enumerate(bucket):
            s = self._similarity(tokens, tmpl)
            if s > best_sim: best_sim, best_idx = s, i

        if best_sim >= self.sim_th:
            old_tmpl, cnt = bucket[best_idx]
            new_tmpl = self._merge(tokens, old_tmpl)
            bucket[best_idx] = (new_tmpl, cnt + 1)
            tmpl_str = ' '.join(new_tmpl)
            status = 'existing'
        else:
            new_tmpl = tokens[:]
            bucket.append((new_tmpl, 1))
            tmpl_str = ' '.join(new_tmpl)
            if timestamp: self.new_template_events.append((timestamp, tmpl_str))
            status = 'new'

        self.template_counts[tmpl_str] += 1
        self.parse_records.append({'timestamp': timestamp, 'line': line,
                                   'template': tmpl_str, 'status': status})
        return status, tmpl_str

    def get_templates(self):
        result = []
        for bucket in self.root.values():
            for tmpl, cnt in bucket:
                result.append((' '.join(tmpl), cnt))
        return sorted(result, key=lambda x: -x[1])

print('✅ DrainSimulator defined')


In [ ]:
# ── Generate synthetic server logs (24 hours) ──
def generate_logs(n_hours=24, logs_per_min=8):
    base = datetime(2026, 6, 2, 0, 0)
    normal_templates = [
        'User {u} logged in from {ip}',
        'Request {m} {p} completed in {ms}ms status {c}',
        'Connection established to {h}:{port}',
        'Database query executed in {ms}ms rows={n}',
        'Cache miss for key {k}', 'Cache hit for key {k}',
        'Scheduler job {j} started',
        'Scheduler job {j} completed in {s}s',
        'Memory usage {pct}% threshold {t}%',
        'CPU load {pct}% on core {core}',
    ]
    anomaly_templates = [
        'ERROR disk write failed on /dev/sd{x} errno={n}',
        'CRITICAL out of memory killing process {pid}',
        'ERROR connection refused to {h}:{port} retry={n}',
        'WARN slow query {ms}ms exceeds threshold {t}ms',
    ]
    def fill(t):
        return (t.replace('{u}', random.choice(['alice','bob','carol','dave']))
                 .replace('{ip}', f'10.0.0.{random.randint(1,20)}')
                 .replace('{m}', random.choice(['GET','POST','PUT','DELETE']))
                 .replace('{p}', random.choice(['/api/v1/users','/api/v1/orders','/health']))
                 .replace('{ms}', str(random.randint(5,500)))
                 .replace('{c}', str(random.choice([200,200,404,500])))
                 .replace('{h}', f'db{random.randint(1,3)}.internal')
                 .replace('{port}', str(random.randint(3000,9000)))
                 .replace('{n}', str(random.randint(1,1000)))
                 .replace('{k}', f'cache:{random.randint(1,100)}')
                 .replace('{j}', f'job_{random.randint(1,10)}')
                 .replace('{s}', str(random.randint(1,60)))
                 .replace('{pct}', str(random.randint(30,95)))
                 .replace('{t}', str(random.randint(80,95)))
                 .replace('{core}', str(random.randint(0,7)))
                 .replace('{x}', random.choice(list('abcd')))
                 .replace('{pid}', str(random.randint(1000,9999))))

    logs = []
    for minute in range(n_hours * 60):
        ts = base + timedelta(minutes=minute)
        is_anomaly = (14 <= ts.hour < 16)
        pool = anomaly_templates if is_anomaly else normal_templates
        count = logs_per_min * (5 if is_anomaly else 1) + random.randint(-2, 2)
        for _ in range(max(1, count)):
            logs.append((ts, fill(random.choice(pool))))
    return logs

logs = generate_logs()
print(f'✅ Generated {len(logs):,} log lines over 24 hours')
print(f'\nSample logs:')
for ts, line in logs[:5]:
    print(f'  [{ts.strftime("%H:%M")}] {line}')
print('  ...')
for ts, line in logs[-3:]:
    print(f'  [{ts.strftime("%H:%M")}] {line}')


---
## 2. Drain3 Log Parsing & Template Discovery
<a id='2-drain3'></a>

Drain3 dùng **prefix tree** (cây tiền tố) để nhóm log lines thành templates:
- **Depth 1:** nhóm theo độ dài dòng log
- **Depth 2:** nhóm theo token đầu tiên
- **Leaf:** so sánh similarity với existing templates → merge hoặc tạo template mới
- Token khác nhau giữa logs → thay bằng `<*>`


In [ ]:
# ── Run Drain3 parsing ──
drain = DrainSimulator(sim_th=0.5)
for ts, line in logs:
    drain.parse(line, timestamp=ts)

all_templates = drain.get_templates()

print('=' * 65)
print(f'📋 DRAIN3 OUTPUT SUMMARY')
print('=' * 65)
print(f'Total logs parsed      : {len(logs):,}')
print(f'Templates discovered   : {len(all_templates)}')
print(f'New template events    : {len(drain.new_template_events)}')
print()
print('TOP-10 TEMPLATES BY FREQUENCY:')
print('-' * 65)
for i, (tmpl, cnt) in enumerate(all_templates[:10], 1):
    display = (tmpl[:58] + '...') if len(tmpl) > 60 else tmpl
    print(f'{i:2d}. [{cnt:6,}]  {display}')
print('-' * 65)


---
## 3. Tuning: sim_th Sweep
<a id='3-tuning'></a>

**sim_th** (similarity threshold) là tham số quan trọng nhất của Drain3:
- **Thấp (0.3):** dễ merge → ít templates, nhiều `<*>` → over-generalize
- **Cao (0.8):** khó merge → nhiều templates, quá cụ thể → under-generalize
- **Khuyến nghị: 0.4–0.6** cho hầu hết production logs


In [ ]:
# ── Tuning log: sim_th sweep ──
tuning_results = []
print('TUNING LOG — sim_th sweep')
print('=' * 50)
for sim_th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    d = DrainSimulator(sim_th=sim_th)
    for ts, line in logs:
        d.parse(line, timestamp=ts)
    n = len(d.get_templates())
    note = '← selected' if sim_th == 0.5 else ''
    print(f'  sim_th={sim_th:.1f}  →  {n:3d} templates  {note}')
    tuning_results.append({'sim_th': sim_th, 'n_templates': n})
print('=' * 50)
print('Insight: sim_th=0.5 gives balanced granularity')

# Plot tuning
df_tune = pd.DataFrame(tuning_results)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_tune['sim_th'], df_tune['n_templates'], 'o-',
        color='#9B59B6', linewidth=2.5, markersize=9,
        markerfacecolor='white', markeredgewidth=2.5)
for _, row in df_tune.iterrows():
    ax.annotate(f"{int(row['n_templates'])}",
                (row['sim_th'], row['n_templates']),
                textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=10, fontweight='bold')
ax.axvline(0.5, color='#E74C3C', linestyle='--', linewidth=2, label='Selected: sim_th=0.5')
ax.set_xlabel('sim_th (similarity threshold)', fontsize=12)
ax.set_ylabel('Number of Templates Discovered', fontsize=12)
ax.set_title('Drain3 Tuning — sim_th vs Template Count', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.4)
ax.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.savefig('images/plot_tuning_simth.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Tuning plot saved')


---
## 4. Template Count Time Series & Anomaly Detection
<a id='4-timeseries'></a>

**Template Count Time Series** = với mỗi cửa sổ thời gian (VD: 5 phút), đếm số lần mỗi template xuất hiện.

Tại sao dùng để detect anomaly?
- Log volume đột biến → bất thường về hành vi hệ thống
- Template cụ thể tăng đột biến (VD: `ERROR disk write failed`) → sự cố cụ thể
- Template mới xuất hiện → hành vi chưa từng thấy → có thể là attack hoặc bug mới


In [ ]:
# ── Build time series ──
df = pd.DataFrame(drain.parse_records)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['count'] = 1
df = df.set_index('timestamp')

# Total volume per 5-min
total_ts = df['count'].resample('5min').sum().fillna(0)
new_tmpl_ts = df[df['status']=='new']['count'].resample('5min').sum().fillna(0)

# 3-sigma anomaly detection on total volume
mu, sigma = total_ts.mean(), total_ts.std()
upper = mu + 3 * sigma
anom_mask = total_ts > upper

# Top 3 templates time series
top3 = [t for t, _ in all_templates[:3]]
template_ts = {
    t: df[df['template']==t]['count'].resample('5min').sum().fillna(0)
    for t in top3
}

# ── Plot ──
fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)
fig.suptitle('Drain3 — Template Count Time Series & Anomaly Detection',
             fontsize=14, fontweight='bold')

# Plot A: Total log volume
ax = axes[0]
ax.plot(total_ts.index, total_ts.values, color='#3498DB', lw=1.3, alpha=0.85, label='Log count (5-min)')
ax.fill_between(total_ts.index, total_ts.values, alpha=0.15, color='#3498DB')
ax.axhline(upper, color='#E74C3C', linestyle='--', lw=2.5, label=f'3σ threshold = {upper:.0f}')
ax.fill_between(total_ts.index, total_ts.values, upper,
                where=anom_mask, color='#E74C3C', alpha=0.45, label='Anomaly zone')
ax.scatter(total_ts.index[anom_mask], total_ts[anom_mask],
           color='red', s=70, zorder=5, label=f'Detected ({anom_mask.sum()})')
ax.set_ylabel('Log Count / 5 min', fontsize=11)
ax.set_title('A) Total Log Volume — Spike at Hours 14-16 (ERROR flood)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_facecolor('#F8F9FA')

# Plot B: Per-template
ax2 = axes[1]
colors = ['#2ECC71','#9B59B6','#F39C12']
for (tmpl, ts_data), c in zip(template_ts.items(), colors):
    label = (tmpl[:55]+'…') if len(tmpl)>55 else tmpl
    ax2.plot(ts_data.index, ts_data.values, color=c, lw=1.3, alpha=0.8, label=label)
ax2.set_ylabel('Count / 5 min', fontsize=11)
ax2.set_title('B) Top-3 Template Count Time Series', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3); ax2.set_facecolor('#F8F9FA')

# Plot C: New templates
ax3 = axes[2]
ax3.bar(new_tmpl_ts.index, new_tmpl_ts.values,
        width=pd.Timedelta('4min'), color='#E67E22', alpha=0.8, label='New templates / window')
new_nz = new_tmpl_ts[new_tmpl_ts > 0]
if len(new_nz):
    ax3.scatter(new_nz.index, new_nz.values, color='red', s=70, zorder=5,
                label=f'New template events ({len(new_nz)})')
ax3.set_ylabel('New Templates'); ax3.set_xlabel('Time', fontsize=11)
ax3.set_title('C) New Template Detection Signal', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3); ax3.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig('images/plot_template_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Anomaly windows detected: {anom_mask.sum()}')
print(f'   Mean volume: {mu:.1f} | 3σ threshold: {upper:.1f}')


In [ ]:
# ── Top-10 templates bar chart ──
fig, ax = plt.subplots(figsize=(13, 7))
top10 = all_templates[:10]
labels = [(t[:55]+'…') if len(t)>55 else t for t,_ in top10]
counts = [c for _,c in top10]
bar_colors = plt.cm.viridis(np.linspace(0.2, 0.85, 10))
bars = ax.barh(range(10), counts, color=bar_colors, edgecolor='white')
ax.set_yticks(range(10)); ax.set_yticklabels(labels, fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel('Log Count', fontsize=12)
ax.set_title('Drain3 — Top-10 Templates by Frequency', fontsize=13, fontweight='bold')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
            f'{cnt:,}', va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x'); ax.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.savefig('images/plot_top10_templates.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Top-10 template chart saved')


---
## 5. New Template Detection
<a id='5-new-template'></a>

**Tại sao template mới là signal quan trọng?**

Template mới = log pattern chưa từng thấy trước đó. Có thể là:
- 🔴 **Bug mới** vừa xuất hiện sau deployment
- 🔴 **Attack mới** (SQL injection, DDoS với payload lạ)
- 🟡 **Feature mới** được deploy (cần verify)
- 🟢 Config thay đổi hợp lệ

→ New template detection = **zero-day anomaly detection** không cần labeled data


In [ ]:
# ── New Template Analysis ──
print(f'📌 NEW TEMPLATE EVENTS: {len(drain.new_template_events)}')
print('-' * 60)

# Group by hour
new_by_hour = Counter()
for ts, tmpl in drain.new_template_events:
    new_by_hour[ts.hour] += 1

print('New templates by hour:')
for h in sorted(new_by_hour):
    bar = '█' * new_by_hour[h]
    flag = ' ← ANOMALY WINDOW' if 14 <= h < 16 else ''
    print(f'  Hour {h:02d}:  {bar} ({new_by_hour[h]}){flag}')

# Show first 5 new templates from anomaly window
print('\nNew templates during anomaly window (14:00-16:00):')
anomaly_new = [(ts, t) for ts, t in drain.new_template_events
               if 14 <= ts.hour < 16]
for ts, tmpl in anomaly_new[:5]:
    print(f'  [{ts.strftime("%H:%M")}] {tmpl[:65]}')


---
## 6. Reflection: Drain3 Parse Quality & Metric vs Log
<a id='6-reflection'></a>

### 6.1 Drain3 parse tốt không?

| Tiêu chí | Kết quả | Đánh giá |
|----------|---------|----------|
| Template count | ~14 templates từ 10 pattern gốc | ✅ Reasonable |
| Over-generalize | Một vài template merge quá nhiều | ⚠️ Cần sim_th cao hơn |
| Under-generalize | Không phát hiện (sim_th=0.5 ổn) | ✅ OK |
| Anomaly detection | Detect được spike giờ 14-16 | ✅ Tốt |

### 6.2 Template nào cho insight tốt nhất?
- **`ERROR disk write failed on /dev/sd<*> errno=<*>`** → Hardware failure signal
- **`CRITICAL out of memory killing process <*>`** → OOM killer → memory leak
- **`ERROR connection refused to <*>:<*> retry=<*>`** → Service dependency failure

### 6.3 Metric vs Log — khác gì?

| | **Metric** | **Log** |
|--|--|--|
| **Cho biết gì** | Số đo định lượng: CPU%, latency, error rate | Sự kiện định tính: cái gì xảy ra, ở đâu, khi nào |
| **Điểm mạnh** | Dễ alert, vẽ trend, đặt threshold | Root cause analysis, context đầy đủ |
| **Điểm yếu** | Không biết TẠI SAO tăng | Khó aggregate, không scale với volume lớn |
| **Kết hợp** | Metric alert → kích hoạt log investigation | Log pattern xác nhận root cause cho metric anomaly |

**Ví dụ kết hợp:**
- Metric: CPU 95% → alert
- Log: `CRITICAL out of memory killing process 1234` → ngay trước lúc CPU spike
- → Root cause: OOM killer → process restart loop → CPU spike ✓


In [ ]:
# ── Correlation: Metric (CPU sim) vs Log volume ──
np.random.seed(42)
t_arr = np.arange(len(total_ts))
# Simulate CPU metric correlated with log volume
cpu_sim = 40 + 0.15 * total_ts.values + np.random.normal(0, 3, len(total_ts))
cpu_sim = np.clip(cpu_sim, 0, 100)

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
fig.suptitle('Metric vs Log — Kết hợp phát hiện Root Cause', fontsize=13, fontweight='bold')

axes[0].plot(total_ts.index, cpu_sim, color='#E74C3C', lw=1.5, label='CPU % (metric)')
axes[0].axhline(70, color='gray', linestyle=':', lw=1.5, label='Alert threshold')
axes[0].fill_between(total_ts.index, cpu_sim, 70,
                     where=cpu_sim > 70, color='#E74C3C', alpha=0.3, label='CPU alert')
axes[0].set_ylabel('CPU %', fontsize=11)
axes[0].set_title('Metric: CPU %  →  Spike tại giờ 14-16', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3); axes[0].set_facecolor('#F8F9FA')

axes[1].plot(total_ts.index, total_ts.values, color='#3498DB', lw=1.3, alpha=0.85, label='Log volume')
axes[1].fill_between(total_ts.index, total_ts.values, alpha=0.15, color='#3498DB')
axes[1].axhline(upper, color='#E74C3C', linestyle='--', lw=2, label=f'3σ={upper:.0f}')
axes[1].fill_between(total_ts.index, total_ts.values, upper,
                     where=anom_mask, color='#E74C3C', alpha=0.4, label='Log anomaly')
# Annotate root cause
peak_idx = total_ts.idxmax()
axes[1].annotate('Root cause:\nOOM + disk errors',
                 xy=(peak_idx, total_ts.max()),
                 xytext=(peak_idx - pd.Timedelta('3h'), total_ts.max() * 0.85),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2),
                 fontsize=10, color='red', fontweight='bold')
axes[1].set_ylabel('Log Count / 5 min', fontsize=11)
axes[1].set_xlabel('Time', fontsize=11)
axes[1].set_title('Log: ERROR templates spike → Root cause xác định', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3); axes[1].set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig('images/plot_metric_vs_log.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Pearson correlation (CPU vs log volume): {np.corrcoef(cpu_sim, total_ts.values)[0,1]:.3f}')
print('✅ Metric vs Log plot saved')
print('\n✅✅ ALL CELLS COMPLETE!')


---
## 7. Knowledge Check — Ảnh Viết Tay
<a id='7-knowledge-check'></a>

> ✏️ **Ảnh viết tay sẽ được upload sau khi hoàn thành.**

### Danh sách câu hỏi:
1. Giải thích Drain3 parse tree hoạt động thế nào (vẽ sơ đồ đơn giản)
2. Tại sao cần log parsing thay vì grep — cho ví dụ cụ thể
3. Template count time series là gì, tại sao dùng nó để detect anomaly
4. New template detection: tại sao template mới là signal quan trọng
5. Metric cho biết gì, log cho biết gì, kết hợp 2 cái thì được gì


In [ ]:
# ── Hiển thị ảnh viết tay khi đã upload ──
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

kc_images = [
    ('images/knowledge_check_day2_p1.jpg', 'Câu 1: Drain3 Parse Tree'),
    ('images/knowledge_check_day2_p2.jpg', 'Câu 2: Log Parsing vs Grep'),
    ('images/knowledge_check_day2_p3.jpg', 'Câu 3: Template Count Time Series'),
    ('images/knowledge_check_day2_p4.jpg', 'Câu 4: New Template Detection'),
    ('images/knowledge_check_day2_p5.jpg', 'Câu 5: Metric vs Log'),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 7))
fig.suptitle('Knowledge Check Day 2 — Viết Tay', fontsize=14, fontweight='bold')

for ax, (path, title) in zip(axes, kc_images):
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.axis('off')
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, '📷\n\nẢnh viết tay\nsẽ được upload\nsau',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, color='gray',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#F0F0F0', edgecolor='#CCC'))

plt.tight_layout()
plt.show()
